[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance1_correction.ipynb)

# Séance 3.1 — Décrire une distribution

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qu'une moyenne décrit — et ce qu'elle ne décrit pas
- choisir entre moyenne et médiane selon la forme de la distribution
- lire un `describe()` ligne par ligne
- mesurer la dispersion avec l'écart-type et l'écart interquartile
- repérer une concentration : quelle part du total tient dans le haut du classement
- distinguer une moyenne pondérée d'une moyenne non pondérée

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

print(cmd.shape)
cmd.head(3)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Moyenne et médiane des quantités

> **Votre mission :**
> - Calculer la moyenne et la médiane de la colonne `qte` (nombre d'unités commandées).
> - Les arrondir à 2 décimales → `moy_qte` et `med_qte`.

In [ ]:
moy_qte = round(cmd["qte"].mean(), 2)     ## 330,77 unites
med_qte = round(cmd["qte"].median(), 2)   ## 195 unites

# Meme phenomene que sur les euros : la moyenne est loin au-dessus
print(moy_qte, "unites en moyenne, mediane a", med_qte)

In [ ]:
verifier("1a - moyenne des quantites", moy_qte == 330.77, "mean()")
verifier("1b - mediane des quantites", med_qte == 195.0, "median()")

### Exercice 2 — Le décompte du haut

> **Votre mission :**
> - Quelle **part** des commandes dépasse la quantité moyenne ? En % arrondi à 1 décimale → `part_sup`.
> - Comparez au 50 % que donnerait une distribution symétrique.

In [ ]:
# (serie > valeur) donne des True/False ; leur moyenne est la proportion
part_sup = round(100 * (cmd["qte"] > cmd["qte"].mean()).mean(), 1)   ## 28,7 %

print(part_sup, "% des commandes depassent la quantite moyenne")

In [ ]:
verifier("2 - part au-dessus de la moyenne", part_sup == 28.7,
         "comparez chaque qte a cmd['qte'].mean(), puis faites la moyenne des True/False")

### Exercice 3 — Un seuil de livraison gratuite

> **Votre mission :**
> - La direction veut offrir la livraison aux **10 % de commandes les plus grosses**.
> - À quel montant faut-il placer le seuil ? → `seuil` (arrondi à 2 décimales)

In [ ]:
# Les 10 % du HAUT commencent au quantile 0.9 : 90 % des commandes
# sont en dessous de ce montant
seuil = round(cmd["ca"].quantile(0.9), 2)   ## 0.9 et non 0.1

print("livraison offerte au-dela de", seuil, "euros")

In [ ]:
verifier("3 - seuil des 10 % du haut", seuil == 1146.28,
         "les 10 % du haut commencent au quantile 0.9, pas 0.1")

### Exercice 4 — Combien de produits différents par commande ?

> **Votre mission :**
> - Calculer l'écart interquartile de `nart` (nombre de produits distincts par commande) → `iqr_nart`.
> - Rappel : écart interquartile = quantile 0.75 − quantile 0.25.

In [ ]:
q1 = cmd["nart"].quantile(0.25)   ## 9 produits distincts
q3 = cmd["nart"].quantile(0.75)   ## 30,5 produits distincts
iqr_nart = q3 - q1                ## la largeur de la moitie centrale

# La moitie centrale des commandes tient entre 9 et 30,5 produits distincts
print("moitie centrale des commandes :", q1, "a", q3, "produits distincts")
print("ecart interquartile :", iqr_nart)

In [ ]:
verifier("4 - ecart interquartile de nart", iqr_nart == 21.5,
         "quantile(0.75) moins quantile(0.25)")

### Exercice 5 — La concentration, version 5 %

> **Votre mission :**
> - Quelle part du chiffre d'affaires les **5 % de commandes les plus grosses** représentent-elles ?
> - En % arrondi à 1 décimale → `part_top5`.

In [ ]:
# ascending=False : les plus grosses en premier
top = cmd["ca"].sort_values(ascending=False)
n5 = int(0.05 * len(cmd))   ## 5 % de 1 955, soit 97 commandes

part_top5 = round(100 * top.head(n5).sum() / top.sum(), 1)   ## leur part du CA
print(n5, "commandes font", part_top5, "% du chiffre d'affaires")

In [ ]:
verifier("5 - part des 5 % du haut", part_top5 == 29.6,
         "triez du plus grand au plus petit, puis divisez par le total")

### Exercice 6 — Le tableau par pays

> **Votre mission :**
> - Construire `parpays` : par pays, l'effectif (`count`), la moyenne et la médiane du `ca`.
> - En extraire la médiane française arrondie à 2 décimales → `med_fr`.

In [ ]:
# agg() prend une liste : trois statistiques d'un seul appel
parpays = cmd.groupby("pays")["ca"].agg(["count", "mean", "median"])

# .loc[ligne, colonne] pour aller chercher une case precise
med_fr = round(parpays.loc["France", "median"], 2)
print(med_fr)

In [ ]:
verifier("6 - mediane francaise", med_fr == 361.35,
         "agg(['count', 'mean', 'median']) puis .loc['France', 'median']")

### Exercice 7 — Le pays le plus déformé

> **Votre mission :**
> - En repartant de `parpays`, garder les pays d'au moins 20 commandes.
> - Ajouter une colonne `ecart` = moyenne − médiane, puis trouver le pays où elle est la plus grande → `pays_ecart`.
> - Cet écart mesure à quel point quelques grosses commandes déforment la moyenne.

In [ ]:
# .copy() avant d'ajouter une colonne a un extrait filtre
gros = parpays.query("count >= 20").copy()   ## on ecarte les petits effectifs
gros["ecart"] = gros["mean"] - gros["median"]   ## la mesure de la deformation

# idxmax() renvoie l'etiquette de la ligne, donc le nom du pays
pays_ecart = gros["ecart"].idxmax()
print(pays_ecart)

In [ ]:
verifier("7 - pays le plus deforme", pays_ecart == "Suede",
         "ecart = mean - median, puis idxmax() pour avoir le nom et non la valeur")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Le directeur commercial veut **une seule phrase** sur le panier des clients pour son comité.
> - Calculer l'écart moyenne − médiane du `ca` → `ecart_ca` (arrondi à 2 décimales).
> - Calculer le rapport écart-type / moyenne — le **coefficient de variation** → `cv` (arrondi à 2 décimales).
> - Puis écrivez la phrase que vous lui donneriez, en commentaire.

In [ ]:
ecart_ca = round(cmd["ca"].mean() - cmd["ca"].median(), 2)   ## 233,84

# std() / mean() : la dispersion rapportee au niveau. Au-dessus de 1,
# l'ecart typique depasse la valeur moyenne elle-meme.
cv = round(cmd["ca"].std() / cmd["ca"].mean(), 2)   ## 1,56 : au-dessus de 1

print("ecart moyenne-mediane :", ecart_ca)
print("dispersion relative   :", cv)

# Une phrase possible :
# "La commande typique est de 356 EUR (mediane). La moyenne de 590 EUR est
#  tiree par une minorite de tres grosses commandes : 10 % d'entre elles
#  font 41 % du chiffre d'affaires. Un seuil commercial doit se caler sur
#  la mediane, pas sur la moyenne."

In [ ]:
verifier("8a - ecart moyenne-mediane", ecart_ca == 233.84,
         "mean() moins median()")
verifier("8b - dispersion relative", cv == 1.56,
         "std() divise par mean() ; au-dessus de 1, la dispersion depasse le niveau moyen")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Un describe() par marché

> **Votre mission :**
> - Produire le `describe()` du `ca` **pour chaque pays**, et n'afficher que les pays d'au moins 20 commandes.
> - Quel pays a la distribution la plus resserrée ? Laquelle la plus étalée ?
> - *Nouveau :* `describe()` s'applique aussi après un `groupby` — `df.groupby('pays')['ca'].describe()`.

In [ ]:
resume = cmd.groupby("pays")["ca"].describe()   ## huit colonnes par pays

resume.query("count >= 20").round(1)   ## sous 20 commandes, on ne commente pas

# Le Royaume-Uni : mediane 301, max 3 161. La Belgique : mediane 349,
# max 1 492. L'Irlande : mediane 658, max 16 775. Trois marches, trois
# regimes — et une seule moyenne globale les melange tous.

### Question 10 — Robuste ou fragile ?

> **Votre mission :**
> - Faire une copie de `cmd`, y multiplier **la plus grosse commande par 10**, puis recalculer moyenne et médiane.
> - Une seule ligne modifiée sur 1 955. De combien chaque indicateur bouge-t-il ?
> - *Rappel :* `.copy()` avant de modifier, sinon vous abîmez la table d'origine.

In [ ]:
faux = cmd.copy()   ## .copy() : on n'abime pas la table d'origine
faux.loc[faux["ca"].idxmax(), "ca"] *= 10   ## UNE ligne sur 1 955

print("moyenne :", round(cmd["ca"].mean(), 2), "->", round(faux["ca"].mean(), 2))
print("mediane :", round(cmd["ca"].median(), 2), "->", round(faux["ca"].median(), 2))

# La moyenne prend 13 % pour UNE ligne sur 1 955. La mediane ne bouge
# pas d'un centime. C'est ca, la robustesse : une erreur de saisie
# unique suffit a fausser un rapport bati sur des moyennes.

### Question 11 — Quel marché est le plus régulier ?

> **Votre mission :**
> - Pour chaque pays d'au moins 20 commandes : l'effectif, la moyenne, l'écart-type, et le **rapport écart-type / moyenne**.
> - Trier par ce rapport. Où l'activité est-elle la plus prévisible ?
> - Ce rapport porte un nom : le **coefficient de variation**.

> 📖 **Le coefficient de variation.** Il ramène l'écart-type au niveau de la
> moyenne, ce qui donne un nombre **sans unité** : on peut alors comparer la
> régularité de marchés dont les paniers n'ont rien à voir. Il ne se lit que sur
> une variable positive dont la moyenne est franchement au-dessus de zéro — sur
> une marge, qui peut être négative, il n'a aucun sens.
> [Wikipédia](https://fr.wikipedia.org/wiki/Coefficient_de_variation)

In [ ]:
regularite = cmd.groupby("pays")["ca"].agg(["count", "mean", "std"])
regularite["cv"] = (regularite["std"] / regularite["mean"]).round(2)   ## sans unite

regularite.query("count >= 20").sort_values("cv").round(1)

# Portugal 0,6 : des commandes de taille comparable, une activite
# previsible — mais sur 28 commandes seulement. La Belgique suit de tres
# pres (0,7) sur 72 commandes : c'est la que la regularite est la mieux
# etablie. Irlande 1,7 : l'ecart-type depasse largement la moyenne, on ne
# peut rien prevoir. Deux marches qu'on ne pilote pas pareil.

### Question 12 — Découper la clientèle en dix

> **Votre mission :**
> - Découper les commandes en **dix groupes de taille égale** selon leur montant (des déciles), puis calculer la part du CA total que pèse chaque décile.
> - Que pèsent les 10 % du haut ? Et les 10 % du bas ?
> - *Nouveau :* `pd.qcut(serie, 10, labels=False)` répartit en dix groupes de même effectif — à ne pas confondre avec `pd.cut`, qui découpe en tranches de même largeur.

In [ ]:
# qcut : dix groupes de MEME EFFECTIF (cut ferait dix tranches de meme largeur)
cmd["decile"] = pd.qcut(cmd["ca"], 10, labels=False) + 1

part = 100 * cmd.groupby("decile")["ca"].sum() / cmd["ca"].sum()
part.round(1)

# Le dernier decile fait 41,4 % du chiffre d'affaires, le premier 0,9 %.
# 196 commandes pesent autant que les 1 542 plus petites reunies.

### Question 13 — La boîte à moustaches

> **Votre mission :**
> - Comparer la distribution du `ca` des quatre pays les plus présents avec une **boîte à moustaches**.
> - Que lisez-vous sur cette figure que le tableau de la question précédente ne montrait pas ?
> - Limitez l'axe à 3 000 € : sinon la commande à 16 775 € écrase les quatre boîtes.
> - *Nouveau :* `df.boxplot(column='ca', by='pays', figsize=(7, 4))`.

> 📖 **La boîte à moustaches.** Elle compare d'un coup d'œil la distribution
> d'une variable entre plusieurs groupes. La boîte va du premier au troisième
> quartile — la **moitié centrale** des commandes — et le trait intérieur est la
> médiane. Chaque moustache s'arrête à la commande la plus éloignée qui reste à
> moins de 1,5 écart interquartile du bord de la boîte : c'est la règle que vous
> avez croisée au bloc 2 pour repérer les valeurs aberrantes. Au-delà, chaque
> commande est tracée seule — des montants rares, pas des erreurs.
> [Wikipédia](https://fr.wikipedia.org/wiki/Bo%C3%AEte_%C3%A0_moustaches)

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

sub.boxplot(column="ca", by="pays", figsize=(7, 4))   ## une boite par pays
plt.ylim(0, 3000)          ## sans quoi un seul point ecrase tout
plt.title("Distribution de ca par pays")
plt.suptitle("")           ## boxplot ajoute un titre automatique, on l'enleve
plt.ylabel("ca : montant de la commande (euros)")
plt.show()

# Une boite a moustaches, c'est un describe() qu'on lit d'un coup d'oeil.

### Question 14 — Le piège du « par jour »

> **Votre mission :**
> - Calculer le chiffre d'affaires total, puis le CA moyen par jour en divisant par 7 jours de semaine.
> - Recommencer en divisant par le nombre de jours **réellement présents** dans le fichier.
> - De combien vous êtes-vous trompé, en pourcentage ?

In [ ]:
par_jour = cmd.groupby("jour")["ca"].sum()

faux = par_jour.sum() / 7                            ## 7 jours supposes
vrai = par_jour.sum() / cmd["jour"].nunique()        ## 6 jours reels

print("en divisant par 7 :", round(faux, 2))
print("par le nombre reel :", round(vrai, 2))
print("erreur :", round(100 * (vrai - faux) / vrai, 1), "%")

# Le samedi n'existe pas dans ce fichier. Diviser par 7 sous-estime
# l'activite quotidienne de 14,3 % — un chiffre faux qui n'a declenche
# aucune alerte.

### Question 15 — Le paragraphe « distribution »

> **Votre mission :**
> - Rédiger, en commentaire, le paragraphe d'ouverture d'une note de direction sur le panier client.
> - Contrainte : quatre phrases maximum, au moins trois chiffres, et **aucune moyenne citée sans sa médiane**.
> - Calculez d'abord les chiffres dont vous avez besoin.

In [ ]:
print("mediane      :", round(cmd["ca"].median(), 2))
print("moyenne      :", round(cmd["ca"].mean(), 2))
print("q1 - q3      :", round(cmd["ca"].quantile(0.25), 2),
      "-", round(cmd["ca"].quantile(0.75), 2))
tri = cmd["ca"].sort_values(ascending=False)
print("part du top 10 % :",
      round(100 * tri.head(int(0.10 * len(cmd))).sum() / tri.sum(), 1), "%")

# Paragraphe possible :
# "La commande typique s'eleve a 356 EUR, et la moitie de nos commandes
#  se situent entre 190 et 660 EUR. La moyenne de 590 EUR, souvent citee,
#  ne decrit aucune commande reelle : elle est tiree par une minorite de
#  gros paniers, les 10 % du haut faisant a eux seuls 41 % du chiffre
#  d'affaires. Tout seuil commercial doit se caler sur la mediane."